<a href="https://colab.research.google.com/github/Jopat2409/com3610_notebooks/blob/main/XLMR_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install torch
%pip install flair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [2]:
import torch
from flair.datasets import CONLL_03
from flair.embeddings import TransformerWordEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer

# 1. get the corpus
corpus = CONLL_03(base_path=".")
print(corpus)

# 2. what label do we want to predict?
label_type = 'ner'

# 3. make the label dictionary from the corpus
label_dict = corpus.make_label_dictionary(label_type=label_type, add_unk=False)
print(label_dict)

# 4. initialize fine-tuneable transformer embeddings WITH document context
embeddings = TransformerWordEmbeddings(
    model='xlm-roberta-large',
    layers="-1",
    subtoken_pooling="first",
    fine_tune=True,
    use_context=True,
)

# 5. initialize bare-bones sequence tagger (no CRF, no RNN, no reprojection)
tagger = SequenceTagger(
    hidden_size=256,
    embeddings=embeddings,
    tag_dictionary=label_dict,
    tag_type='ner',
    use_crf=False,
    use_rnn=False,
    reproject_embeddings=False,
)

# 6. initialize trainer
trainer = ModelTrainer(tagger, corpus)

# 7. run fine-tuning
trainer.fine_tune(
    'resources/taggers/ner-english-large',
    learning_rate=5.0e-6,
    mini_batch_size=4,
    mini_batch_chunk_size=1,
    max_epochs=5,
    weight_decay=0.,
    save_final_model=True
)

2025-03-13 03:59:16,737 Reading data from conll_03
2025-03-13 03:59:16,737 Train: conll_03/train.txt
2025-03-13 03:59:16,738 Dev: conll_03/dev.txt
2025-03-13 03:59:16,738 Test: conll_03/test.txt
Corpus: 14903 train + 3449 dev + 3658 test sentences
2025-03-13 03:59:23,718 Computing label dictionary. Progress:


1it [00:00, 1350.82it/s]
14903it [00:00, 35989.35it/s]

2025-03-13 03:59:24,184 Dictionary created for label 'ner' with 4 values: LOC (seen 8258 times), ORG (seen 7100 times), PER (seen 6527 times), MISC (seen 1681 times)


Dictionary with 4 tags: LOC, ORG, PER, MISC


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

2025-03-13 09:42:28,302 DEV : loss 0.06916142255067825 - f1-score (micro avg)  0.9722


2025-03-13 09:42:35,280 ----------------------------------------------------------------------------------------------------
2025-03-13 09:42:35,283 Testing using last state of model ...


100%|██████████| 229/229 [00:41<00:00,  5.52it/s]

2025-03-13 09:43:16,794 
Results:
- F-score (micro) 0.9697
- F-score (macro) 0.9525
- Accuracy 0.9575

By class:
              precision    recall  f1-score   support

         ORG     0.9706    0.9676    0.9691      1946
         LOC     0.9660    0.9714    0.9687      1784
         PER     0.9962    0.9950    0.9956      1591
        MISC     0.8578    0.8960    0.8765       404

   micro avg     0.9680    0.9714    0.9697      5725
   macro avg     0.9477    0.9575    0.9525      5725
weighted avg     0.9683    0.9714    0.9698      5725

2025-03-13 09:43:16,794 ----------------------------------------------------------------------------------------------------


{'test_score': 0.9696599825632084}

In [8]:
from flair.datasets import DataLoader

tagger = SequenceTagger.load('./resources/taggers/ner-english-large/final-model.pt')

result = tagger.evaluate(corpus.test, 'ner', out_path="predictions.txt")
print(result)

2025-03-13 11:11:11,035 SequenceTagger predicts: Dictionary with 17 tags: O, S-LOC, B-LOC, E-LOC, I-LOC, S-ORG, B-ORG, E-ORG, I-ORG, S-PER, B-PER, E-PER, I-PER, S-MISC, B-MISC, E-MISC, I-MISC


100%|██████████| 115/115 [00:40<00:00,  2.81it/s]


Results:
- F-score (micro) 0.9697
- F-score (macro) 0.9525
- Accuracy 0.9575

By class:
              precision    recall  f1-score   support

         ORG     0.9706    0.9676    0.9691      1946
         LOC     0.9660    0.9714    0.9687      1784
         PER     0.9962    0.9950    0.9956      1591
        MISC     0.8578    0.8960    0.8765       404

   micro avg     0.9680    0.9714    0.9697      5725
   macro avg     0.9477    0.9575    0.9525      5725
weighted avg     0.9683    0.9714    0.9698      5725

Loss: 0.08049154281616211'
